# CS336 TinyStories — GPU training

Snapshot of the local assignment implementation. Trains a 4-layer, 128-dimensional model with a 4096-token BPE vocabulary on TinyStories. The model run stops at 6000 updates or 10 minutes. Data preparation runs first and has progress output.

Use **Runtime → Change runtime type → T4 GPU**, then **Run all**. No Drive mounting or repository credentials are required. Download the result archive at the end before disconnecting the runtime.


In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from pathlib import Path

def run_live(command, check=True):
    """Stream stdout and stderr into Colab instead of losing child logs."""
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            status = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if check and status:
        raise subprocess.CalledProcessError(status, command)
    return subprocess.CompletedProcess(command, status)

ROOT = Path('/content/assignment1-basics')
ROOT.mkdir(parents=True, exist_ok=True)
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '4d7f85c2b7fce0eb9d2951b33205a7f0ea72db34f2decd894fca1e11c5a488d5'
with zipfile.ZipFile(io.BytesIO(payload)) as bundle:
    bundle.extractall(ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
run_live([sys.executable, '-m', 'pip', 'install', '-q', 'regex', 'numpy'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU in Runtime → Change runtime type, then rerun.'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('PyTorch:', torch.__version__, 'Source SHA256:', hashlib.sha256(payload).hexdigest(), flush=True)
run_live(['nvidia-smi'], check=True)


In [ ]:
# Download directly to Colab; your local multi-GB dataset is not uploaded.
DATA = ROOT / 'data'
DATA.mkdir(exist_ok=True)
for split in ('train', 'valid'):
    name = f'TinyStoriesV2-GPT4-{split}.txt'
    url = f'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/{name}'
    print(f'Downloading {name}', flush=True)
    run_live(['wget', '--continue', '--progress=dot:giga', '-O', str(DATA/name), url], check=True)
    print(name, (DATA/name).stat().st_size, 'bytes', flush=True)


In [ ]:
# Train BPE on training text only, then encode both full splits.
# BPE counting reports each ~64 MiB; encoding reports token totals.
workers = max(1, min(4, os.cpu_count() or 1))
run_live([sys.executable, '-u', '-m', 'cs336_basics.prepare_data',
    '--train-text', str(DATA/'TinyStoriesV2-GPT4-train.txt'),
    '--valid-text', str(DATA/'TinyStoriesV2-GPT4-valid.txt'),
    '--output-dir', str(DATA/'tinystories'), '--vocab-size', '4096',
    '--workers', str(workers)], check=True)


In [ ]:
# Run the existing small-model configuration on CUDA.
RUN = ROOT / 'runs' / 'tinystories-colab'
command = [sys.executable, '-u', '-m', 'cs336_basics.train',
    '--train-data', str(DATA/'tinystories/train.bin'),
    '--valid-data', str(DATA/'tinystories/valid.bin'),
    '--config', 'configs/tinystories_small.json', '--device', 'cuda',
    '--output-dir', str(RUN)]
if (RUN/'latest.pt').exists():
    command += ['--resume', str(RUN/'latest.pt')]
run_live(command, check=True)


In [ ]:
# Generate examples and preserve the checkpoint, tokenizer, metrics, and source.
run_live([sys.executable, '-u', '-m', 'cs336_basics.generate',
    '--checkpoint', str(RUN/'latest.pt'), '--tokenizer-dir', str(DATA/'tinystories'),
    '--device', 'cuda', '--max-new-tokens', '200', '--output', str(RUN/'samples.json')], check=True)
import shutil
EXPORT = Path('/content/cs336-tinystories-colab-results.zip')
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in (RUN, ROOT/'cs336_basics', ROOT/'configs'):
        for path in sorted(folder.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                archive.write(path, path.relative_to(ROOT))
    for path in sorted((DATA/'tinystories').glob('*.json')):
        archive.write(path, path.relative_to(ROOT))
rows = [json.loads(line) for line in (RUN/'metrics.jsonl').read_text().splitlines()]
print('First logged metrics:', rows[0])
print('Final logged metrics:', rows[-1])
print('Results archive:', EXPORT, EXPORT.stat().st_size, 'bytes')
from IPython.display import display, FileLink
display(FileLink(str(EXPORT)))


In [ ]:
# Download a copy before disconnecting the runtime.
from google.colab import files
files.download(str(EXPORT))
